# Agente Strands com Observabilidade OpenLIT no Amazon Bedrock AgentCore Runtime

## Visão Geral

Este notebook demonstra como implantar um agente Strands no Amazon Bedrock AgentCore Runtime com integração de observabilidade OpenLIT. A implementação utiliza modelos Amazon Bedrock Claude e envia dados de telemetria para o [OpenLIT](https://github.com/openlit/openlit) através do OpenTelemetry (OTEL).

## Componentes Principais

- **Strands Agents**: Framework Python para construir agentes baseados em LLM com suporte a telemetria integrado
- **Amazon Bedrock AgentCore Runtime**: Serviço de runtime gerenciado para hospedar e escalar agentes na AWS
- **OpenLIT**: Plataforma de observabilidade open-source para aplicações LLM e AI Agents construída sobre OpenTelemetry
- **OpenTelemetry**: Protocolo padrão da indústria para coleta e exportação de dados de telemetria

## Arquitetura

O agente é containerizado e implantado no AgentCore Runtime, que fornece endpoints HTTP para invocação. Os dados de telemetria fluem do agente Strands através dos exporters OTEL para o OpenLIT para monitoramento e depuração. A implementação desabilita a observabilidade padrão do AgentCore para utilizar o OpenLIT em seu lugar.

## Pré-requisitos

- Python 3.10+
- Credenciais AWS configuradas com permissões do Bedrock e AgentCore
- [OpenLIT](https://github.com/openlit/openlit) implantado
- Docker instalado localmente
- Acesso aos modelos Amazon Bedrock Claude em us-west-2

## Configuração do OpenLIT

Antes de prosseguir com a implantação do agente, você precisa configurar o OpenLIT para receber dados de telemetria. O OpenLIT deve estar acessível a partir do Amazon Bedrock AgentCore Runtime.

### Opções de Implantação

Você tem duas opções principais para implantar o OpenLIT:

#### Opção 1: Implantação com Docker (Mais Rápida para Testes)
Implante o OpenLIT usando Docker Compose. Esta é a abordagem mais simples para começar:

```bash
# Usando Docker Compose (recomendado para configuração rápida)
git clone https://github.com/openlit/openlit.git
cd openlit
docker compose up -d
```

Isso iniciará o OpenLIT em `http://localhost:3000` (UI) e `http://localhost:4318` (endpoint OTEL). 

Para que o AgentCore possa acessá-lo, você precisa implantá-lo em uma máquina (por exemplo, instância EC2) que o AgentCore consiga alcançar:
1. Implante em uma instância EC2 ou serviço de contêiner com IP público
2. Certifique-se de que a porta 4318 esteja acessível (configure security groups para permitir tráfego de entrada)
3. Use a URL do endpoint público (por exemplo, `http://<ec2-public-ip>:4318`) na configuração do seu AgentCore

#### Opção 2: Implantação com Kubernetes (Pronta para Produção)
Para uso em produção, implante o OpenLIT no Kubernetes usando Helm:

```bash
# Adicionar repositório Helm do OpenLIT
helm repo add openlit https://openlit.github.io/helm-charts
helm repo update

# Instalar OpenLIT
helm install openlit openlit/openlit
```

**Configuração Padrão:**
- Por padrão, o OpenLIT cria um serviço LoadBalancer com IP público
- Isso torna o endpoint OTEL acessível publicamente em `http://<load-balancer-ip>:4318`
- A UI estará acessível em `http://<load-balancer-ip>:3000`

**Configuração VPC/Privada:**
Se você preferir manter o OpenLIT privado dentro de uma VPC (recomendado para produção):

1. **Implante na mesma VPC que o AgentCore** ou configure VPC peering
2. **Altere o tipo de serviço para ClusterIP ou use Load Balancer interno:**
   ```bash
   helm install openlit openlit/openlit \
     --set service.type=ClusterIP
   ```
   Ou para load balancer interno da AWS:
   ```bash
   helm install openlit openlit/openlit \
     --set service.annotations."service\.beta\.kubernetes\.io/aws-load-balancer-internal"="true"
   ```
3. **Configure security groups/network policies** para permitir tráfego entre AgentCore e OpenLIT
4. Use o endpoint interno (por exemplo, `http://openlit.default.svc.cluster.local:4318` ou DNS do load balancer interno)

### Obtendo seu Endpoint OpenLIT

Após a implantação do OpenLIT, você usará o endpoint OTEL no formato:
- **Docker com IP público**: `http://<ec2-public-ip-or-domain>:4318`
- **Kubernetes com LoadBalancer público**: `http://<load-balancer-external-ip>:4318`
- **Kubernetes com interno/VPC**: `http://<internal-dns-or-ip>:4318`

📝 **Salve esta URL do endpoint** - você precisará dela na etapa de configuração abaixo.

Para instruções detalhadas de implantação, consulte o [Guia de Instalação do OpenLIT](https://docs.openlit.io/latest/openlit/installation).

## Instalação

Instale as dependências necessárias a partir do arquivo requirements.txt:

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Implementação do Agente

O arquivo do agente (`strands_claude.py`) implementa um agente de viagens com capacidades de busca na web. A configuração principal inclui:
- Inicialização da telemetria Strands com exporter OTLP
- Uso de inicialização lazy para garantir que as variáveis de ambiente sejam carregadas

In [ ]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Function to initialize Bedrock model
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Initialize the Bedrock model
bedrock_model = get_bedrock_model()

# Define the agent's system prompt
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """Initialize the agent with proper telemetry configuration."""

    # Initialize Strands telemetry with 3P configuration
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    
    # Create and cache the agent
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # Initialize agent with proper configuration
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

### Configurar implantação no AgentCore Runtime

Em seguida, usaremos nosso starter toolkit para configurar a implantação no AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR no lançamento.

Durante a etapa de configuração, seu Dockerfile será gerado com base no código da sua aplicação. Observe que ao usar o `bedrock_agentcore_starter_toolkit` para configurar seu agente, ele configura a Observabilidade do AgentCore por padrão, portanto, para usar o OpenLIT, você precisa remover a configuração de Observabilidade do AgentCore conforme explicado abaixo:

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_openlit_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response

## Implantar no AgentCore Runtime

Agora que temos um Dockerfile, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
# OpenLIT configuration
# IMPORTANT: Replace with your actual OpenLIT OTEL endpoint
# Examples:
#   - Public EC2: "http://ec2-XX-XXX-XXX-XXX.compute-1.amazonaws.com:4318"
#   - VPC/Private: "http://10.0.1.100:4318" or "http://openlit.internal:4318"

otel_endpoint = "http://<your-openlit-host>:4318"  # ⚠️ REPLACE THIS with your OpenLIT endpoint

env_vars = {
    "BEDROCK_MODEL_ID": "us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # Example model ID
    "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use OpenLIT OTEL endpoint
    "DISABLE_ADOT_OBSERVABILITY": "true",
}

launch_result = agentcore_runtime.launch(env_vars=env_vars)
launch_result

## Verificar Status da Implantação

Aguarde o runtime ficar pronto antes de invocar:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invocando o AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "I'm planning a weekend trip to Tokyo. What are the must-visit places and local food I should try?"})

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

## Visualizar Traces no OpenLIT

Para visualizar os traces:
1. Acesse seu painel do OpenLIT:
   - Para self-hosted: Navegue até `http://your-openlit-host:3000`
2. Clique na seção "Requests"
3. Filtre pelo nome do seu serviço

Os traces incluirão:
- Detalhes de invocação do agente com contexto completo de requisição/resposta
- Chamadas de ferramentas (busca na web) com tempo de execução
- Interações com o modelo com latência, uso de tokens e estimativas de custo
- Payloads de requisição/resposta
- Rastreamento de erros e informações de depuração
- Métricas de desempenho e analytics

## Limpeza (Opcional)

Limpe os recursos implantados:

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Resumo

Você implantou com sucesso um agente Strands no Amazon Bedrock AgentCore Runtime com observabilidade OpenLIT. A implementação demonstra:
- Integração de agentes Strands com o AgentCore Runtime
- Configuração do OpenTelemetry para enviar traces ao OpenLIT
- Ordem de inicialização adequada para garantir a configuração de telemetria
- Invocação através de SDK e cliente boto3

O agente agora está rodando em um ambiente gerenciado e escalável com observabilidade completa através do OpenLIT. O OpenLIT fornece monitoramento abrangente incluindo:
- Visualização de traces em tempo real
- Rastreamento e análise de custos
- Métricas de desempenho
- Rastreamento de erros e depuração
- Analytics de uso de tokens